<a href="https://colab.research.google.com/github/PercyTechX/del-excel-al-ml/blob/main/temporada-2/03-random-forest/solucion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Solución 2.3 · Random Forest: ¿vale la pena?

**Serie:** Del Excel al Machine Learning · **Temporada 2** · 🎬 Video próximamente

Estas son las respuestas del ejercicio. Si la tuya es distinta pero da el mismo resultado, ¡también está bien! En programación suele haber más de una forma de llegar a la respuesta.

### Prepara los datos

Ejecuta estas celdas. Dejan listas `entreno`, `prueba`, `x_entreno` y `x_prueba` con despachos, y `cred_entreno`, `cred_prueba`, `x_cred_entreno`, `x_cred_prueba` y `mes` con créditos. Deben quedar `5408` y `1531` créditos.

Algunos ejercicios usan variables de los anteriores, así que **ejecútalos en orden**.

In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

DATOS = "https://raw.githubusercontent.com/PercyTechX/del-excel-al-ml/main/datos/"

In [2]:
def preparar_despachos(despachos):
    """Deja los datos que se conocen al preparar un despacho: textos como 0 y 1, mes y día."""
    fecha = pd.to_datetime(despachos["fecha"])
    x = pd.get_dummies(despachos[["ciudad", "tipo", "transportista"]], dtype=int)
    x = x.drop(columns=["ciudad_Lima", "tipo_Bodega", "transportista_Propio"], errors="ignore")
    x["mes"] = fecha.dt.month
    x["dia"] = fecha.dt.dayofweek
    return x.join(despachos[["distancia_km", "lineas", "presentaciones"]])

In [3]:
def parte_encontrada(periodo, real, probabilidad, vigilar=0.10):
    """Parte de los casos reales hallada al vigilar, en cada periodo, a los de mayor riesgo."""
    orden = pd.DataFrame({"periodo": periodo, "real": real, "probabilidad": probabilidad})
    orden["azar"] = np.random.default_rng(0).random(len(orden))
    orden = orden.sort_values(["periodo", "probabilidad", "azar"], ascending=[True, False, True])
    puesto = orden.groupby("periodo").cumcount()
    cupo = (orden.groupby("periodo")["real"].transform("size") * vigilar).round().clip(lower=1)
    return orden.loc[puesto < cupo, "real"].sum() / orden["real"].sum()

In [4]:
def preparar_creditos(creditos):
    """Agrega las variables del modelo y separa entrenamiento y prueba al 1 de julio de 2025."""
    tabla = creditos.assign(
        log_monto=np.log(creditos["monto_sobre_limite"]),
        tasa_previa=(creditos["morosos_previos"] / creditos["casos_previos"]).fillna(0),
        sin_historial=(creditos["casos_previos"] == 0).astype(int),
    )
    fecha = pd.to_datetime(tabla["fecha"])
    se_sabe = fecha + pd.to_timedelta(tabla["plazo_dias"] + 31, unit="D")
    con_resultado = tabla["moroso"].notna()
    corte = pd.Timestamp("2025-07-01")
    return tabla[(se_sabe <= corte) & con_resultado], tabla[(fecha >= corte) & con_resultado]

In [5]:
def preparar_x(tabla, numeros, textos, columnas=None):
    """Convierte los textos en columnas de 0 y 1. Con `columnas`, usa las de entrenamiento."""
    x = pd.get_dummies(tabla[numeros + textos], columns=textos, dtype=float)
    if columnas is not None:
        x = x.reindex(columns=columnas, fill_value=0)
    return x

In [6]:
despachos = pd.read_csv(DATOS + "despachos.csv")

entreno = despachos[despachos["fecha"] < "2025-07-01"]
prueba = despachos[despachos["fecha"] >= "2025-07-01"]

x_entreno = preparar_despachos(entreno)
x_prueba = preparar_despachos(prueba)

In [7]:
creditos = pd.read_csv(DATOS + "creditos.csv")
cred_entreno, cred_prueba = preparar_creditos(creditos)

numeros = ["log_monto", "antiguedad_anios", "tasa_previa", "sin_historial"]
textos = ["tipo"]
x_cred_entreno = preparar_x(cred_entreno, numeros, textos)
x_cred_prueba = preparar_x(cred_prueba, numeros, textos, x_cred_entreno.columns)

mes = cred_prueba["fecha"].str[:7]
print(len(cred_entreno), len(cred_prueba))

5408 1531


### 1. Tu bosque

Entrena un bosque de 100 árboles con `x_entreno`, con `min_samples_leaf=20` y `random_state=0`, en `bosque`. ¿Qué parte de los tardíos encuentra en `prueba`? Debe dar `0.29`.

In [8]:
bosque = RandomForestClassifier(n_estimators=100, min_samples_leaf=20, random_state=0, n_jobs=-1)
bosque.fit(x_entreno, entreno["llego_tarde"])

prob = bosque.predict_proba(x_prueba)[:, 1]
print(round(parte_encontrada(prueba["fecha"], prueba["llego_tarde"], prob), 3))

0.29


### 2. Las tres más importantes

Muestra las tres columnas de mayor importancia en `bosque`.

In [9]:
pd.Series(bosque.feature_importances_, index=x_entreno.columns).sort_values().tail(3)

distancia_km             0.155583
transportista_Tercero    0.193977
dia                      0.290985
dtype: float64

### 3. ¿Importancia o causa?

Responde en un comentario. `tipo_Minimarket` tiene muy poca importancia. ¿Eso prueba que el tipo de cliente no importa nunca en logística? ¿Por qué?

In [10]:
# No. Solo dice que este bosque, con estos datos, casi no usó esa columna.
# Ninguna regla usa el tipo directamente, y aun así los minimarkets se atrasan algo más.
# Poca importancia no significa que no haya diferencias, y una empresa real podría ser distinta.

### 4. Créditos: los tres modelos

Entrena con `x_cred_entreno` y `cred_entreno["moroso"]`:

- `make_pipeline(StandardScaler(), LogisticRegression())`
- `DecisionTreeClassifier(max_depth=3, random_state=0)`
- `RandomForestClassifier(n_estimators=200, min_samples_leaf=20, random_state=0, n_jobs=-1)`

Calcula para cada uno qué parte de los morosos encuentra vigilando el 10 % de cada mes.

> 💡 `parte_encontrada(mes, cred_prueba["moroso"], prob)`. La regresión debe dar `0.296`.

In [11]:
modelos = {
    "regresion": make_pipeline(StandardScaler(), LogisticRegression()),
    "arbol": DecisionTreeClassifier(max_depth=3, random_state=0),
    "bosque": RandomForestClassifier(
        n_estimators=200, min_samples_leaf=20, random_state=0, n_jobs=-1
    ),
}
for nombre, modelo in modelos.items():
    modelo.fit(x_cred_entreno, cred_entreno["moroso"])
    prob = modelo.predict_proba(x_cred_prueba)[:, 1]
    print(nombre, round(parte_encontrada(mes, cred_prueba["moroso"], prob), 3))

regresion 0.296
arbol 0.26


bosque 0.278


### 5. Un árbol sin límite, en créditos

Haz lo mismo con `DecisionTreeClassifier(random_state=0)`. ¿Qué parte encuentra? Debe dar `0.126`.

In [12]:
sin_limite = DecisionTreeClassifier(random_state=0)
sin_limite.fit(x_cred_entreno, cred_entreno["moroso"])

prob = sin_limite.predict_proba(x_cred_prueba)[:, 1]
print(round(parte_encontrada(mes, cred_prueba["moroso"], prob), 3))

0.126


### 6. ¿Cuál usarías?

Responde en un comentario. Con estos resultados, ¿qué modelo usarías para ordenar los créditos, y por qué? Son pocos morosos: 223 en la prueba.

In [13]:
# Regresión, árbol y bosque quedan a la par: unas pocas compras de diferencia.
# Yo usaría la regresión o el árbol pequeño, porque se explican. El bosque no compensa aquí.
# El árbol sin límite es el peor: se aprendió las respuestas.

## 🎉 ¡Bien hecho!

Nos vemos en el **episodio 2.4: Series de tiempo: día de la semana y feriados**.

---

*Del Excel al Machine Learning* · por [Percy Nazario](https://www.linkedin.com/in/percy-nazario/) · [Repositorio](https://github.com/PercyTechX/del-excel-al-ml) · Contenido bajo licencia CC BY 4.0